In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.models as models
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

# Ensure deterministic behavior for forensic auditability
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# Verify GPU availability
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Forensic Engine Initialization: Running on {DEVICE}")
print(f"Torch version: {torch.__version__} | Torchaudio version: {torchaudio.__version__}")

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# =====================================================================
# THE MASTER DATA PIPELINE (Config, Smart Scanner, & PyTorch Dataset)
# =====================================================================
class UniversalForensicConfig:
    # Your verified Kaggle paths
    DATASET_PATHS = {
        "dfdc": "/kaggle/input/datasets/wuliaokaola/dfdc-audio", 
        "in_the_wild": "/kaggle/input/datasets/bhaveshkumars/release-in-the-wild", 
        "wavefake": "/kaggle/input/datasets/walimuhammadahmad/fakeaudio", 
        "asvspoof_2021": "/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021"
    }
    
    # Audio Processing Hyperparameters
    SAMPLE_RATE = 16000
    DURATION_SECONDS = 4
    TARGET_SAMPLES = SAMPLE_RATE * DURATION_SECONDS
    N_MELS = 128  
    
    # Model Training Hyperparameters
    BATCH_SIZE = 32
    EPOCHS = 15
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY = 1e-4
    SAVE_PATH = "multi_dataset_forensic_resnet.pt"

config = UniversalForensicConfig()

def build_master_dataframe():
    print("--- Starting Multi-Dataset Aggregation ---")
    all_records = []
    
    for dataset_name, base_path in config.DATASET_PATHS.items():
        if not os.path.exists(base_path):
            print(f"Warning: Directory not found for {dataset_name} at {base_path}")
            continue
            
        # 1. Grab ALL possible audio formats (.wav, .flac, .aac)
        audio_files = glob.glob(os.path.join(base_path, "**/*.wav"), recursive=True) + \
                      glob.glob(os.path.join(base_path, "**/*.flac"), recursive=True) + \
                      glob.glob(os.path.join(base_path, "**/*.aac"), recursive=True)
                      
        real_count, fake_count = 0, 0
        
        # 2. Pre-load ASVspoof 2021 labels by hunting for .txt key files
        asv_labels = {}
        if 'asvspoof' in dataset_name.lower():
            txt_files = glob.glob(os.path.join(base_path, "**/*.txt"), recursive=True)
            for txt in txt_files:
                try:
                    with open(txt, 'r') as f:
                        for line in f:
                            line = line.strip()
                            if 'bonafide' in line.lower() or 'spoof' in line.lower():
                                parts = line.split()
                                target = 0 if 'bonafide' in line.lower() else 1
                                for p in parts:
                                    if len(p) > 5 and ('_' in p or p.isalnum()):
                                        asv_labels[p.replace('.flac', '').replace('.wav', '')] = target
                except Exception: pass
                
        # 3. Pre-load DFDC labels by hunting for metadata.json
        dfdc_labels = {}
        if 'dfdc' in dataset_name.lower():
            json_files = glob.glob(os.path.join(base_path, "**/*.json"), recursive=True)
            for jf in json_files:
                try:
                    df_json = pd.read_json(jf).T
                    for index, row in df_json.iterrows():
                        base_name = index.replace('.mp4', '').replace('.aac', '')
                        dfdc_labels[base_name] = 1 if row.get('label', '').upper() == 'FAKE' else 0
                except Exception: pass

        # 4. Process all discovered audio files
        for file_path in audio_files:
            file_path_lower = file_path.lower()
            filename_stem = os.path.splitext(os.path.basename(file_path))[0]
            target = None
            
            # Rule A: WaveFake (100% Fake)
            if 'wavefake' in dataset_name.lower():
                target = 1
                
            # Rule B: In-The-Wild (Folder based: /real/ or /fake/)
            elif '/real/' in file_path_lower or '\\real\\' in file_path_lower:
                target = 0
            elif '/fake/' in file_path_lower or '\\fake\\' in file_path_lower:
                target = 1
                
            # Rule C: ASVspoof 2021 (Dictionary lookup from txt files)
            elif 'asvspoof' in dataset_name.lower():
                target = asv_labels.get(filename_stem, None)
                
            # Rule D: DFDC (Dictionary lookup from json)
            elif 'dfdc' in dataset_name.lower():
                target = dfdc_labels.get(filename_stem, None)
                
            # Append if we successfully labeled it
            if target is not None:
                if target == 0: real_count += 1
                else: fake_count += 1
                all_records.append({'filepath': file_path, 'target': target})
                
        print(f"-> {dataset_name.upper()}: Processed {real_count} real and {fake_count} fake samples.")
        if real_count == 0 and fake_count == 0:
            print(f"   [!] WARNING: Could not find explicit labels for {dataset_name}. Skipping these files to protect model integrity.")

    master_df = pd.DataFrame(all_records)
    
    # Safety Guard
    class_counts = master_df['target'].value_counts().to_dict() if not master_df.empty else {}
    real_count = class_counts.get(0, 0)
    fake_count = class_counts.get(1, 0)
    
    print("\n--- Final Summary Data Split ---")
    print(f"Total REAL samples (Class 0): {real_count}")
    print(f"Total FAKE samples (Class 1): {fake_count}")
    
    if real_count == 0 or fake_count == 0:
        raise ValueError(
            f"CRITICAL ERROR: Cannot train. One of your classes has 0 files! "
            f"(Real: {real_count}, Fake: {fake_count})."
        )
        
    return master_df

class UniversalDeepfakeDataset(Dataset):
    def __init__(self, df, config, is_train=True):
        self.df = df
        self.config = config
        self.is_train = is_train
        
        self.mel_transform = T.MelSpectrogram(
            sample_rate=config.SAMPLE_RATE,
            n_mels=config.N_MELS,
            n_fft=1024,
            hop_length=256,
            f_min=20,
            f_max=8000
        )
        self.db_transform = T.AmplitudeToDB()
        self.freq_masking = T.FrequencyMasking(freq_mask_param=15)
        self.time_masking = T.TimeMasking(time_mask_param=35)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filepath = self.df.iloc[idx]['filepath']
        label = self.df.iloc[idx]['target']
        
        try:
            # Torchaudio handles .wav, .flac, and usually .aac if ffmpeg backend is active
            waveform, sr = torchaudio.load(filepath)
            
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
                
            if sr != self.config.SAMPLE_RATE:
                waveform = T.Resample(sr, self.config.SAMPLE_RATE)(waveform)
                
            if waveform.shape[1] > self.config.TARGET_SAMPLES:
                if self.is_train:
                    start = torch.randint(0, waveform.shape[1] - self.config.TARGET_SAMPLES, (1,)).item()
                    waveform = waveform[:, start:start + self.config.TARGET_SAMPLES]
                else:
                    start = (waveform.shape[1] - self.config.TARGET_SAMPLES) // 2
                    waveform = waveform[:, start:start + self.config.TARGET_SAMPLES]
            elif waveform.shape[1] < self.config.TARGET_SAMPLES:
                padding = self.config.TARGET_SAMPLES - waveform.shape[1]
                waveform = F.pad(waveform, (0, padding))
                
            log_mel = self.db_transform(self.mel_transform(waveform))
            if self.is_train:
                log_mel = self.freq_masking(log_mel)
                log_mel = self.time_masking(log_mel)
                
            return log_mel, torch.tensor(label, dtype=torch.float32)
            
        except Exception as e:
            # Safe fallback if an .aac file is corrupted or unsupported by Kaggle's backend
            dummy = torch.zeros((1, self.config.N_MELS, self.config.TARGET_SAMPLES // 256 + 1))
            return dummy, torch.tensor(label, dtype=torch.float32)

# Execute Pipeline
master_df = build_master_dataframe()

train_df, val_df = train_test_split(master_df, test_size=0.15, stratify=master_df['target'], random_state=42)

print("\nInitializing PyTorch Datasets...")
train_dataset = UniversalDeepfakeDataset(train_df, config, is_train=True)
val_dataset = UniversalDeepfakeDataset(val_df, config, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print(f"Data Pipeline Ready. Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

In [ ]:
class ForensicAudioResNet(nn.Module):
    def __init__(self):
        super(ForensicAudioResNet, self).__init__()
        # Initialize standard deep ResNet-18 backbone
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # Modify the input layer: Audio Spectrograms are 1-channel, ImageNet expects 3-channel (RGB)
        original_conv = self.backbone.conv1
        self.backbone.conv1 = nn.Conv2d(
            in_channels=1, 
            out_channels=original_conv.out_channels, 
            kernel_size=original_conv.kernel_size, 
            stride=original_conv.stride, 
            padding=original_conv.padding, 
            bias=False
        )
        
        # Initialize the new channel with the average structural weights of pre-trained channels
        with torch.no_grad():
            self.backbone.conv1.weight.data = original_conv.weight.data[:, 0:1, :, :].clone()
            
        # Replace projection layer with a forensic head incorporating Dropout to mitigate overfitting
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 1) # Raw Logit output
        )

    def forward(self, x):
        return self.backbone(x)

model = ForensicAudioResNet().to(DEVICE)
print("Model initialized and adapted for 1-Channel Log-Mel input features.")

In [ ]:
def compute_equal_error_rate(labels, scores):
    """
    Calculates the Equal Error Rate (EER), the specific threshold point 
    where False Acceptance Rate (FAR) equals False Rejection Rate (FRR).
    """
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    
    # Locate index where absolute variance between FPR and FNR approaches zero
    absolute_diff = np.abs(fnr - fpr)
    eer_idx = np.nanargmin(absolute_diff)
    
    eer = fpr[eer_idx]
    optimal_threshold = thresholds[eer_idx]
    return eer, optimal_threshold

In [ ]:
# Compute loss weighting factor due to extreme class imbalance in ASVspoof data
# (Approximately 9 deepfakes to every 1 real sample)
pos_weight_factor = torch.tensor([9.0]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_factor)

optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.EPOCHS)

best_eer = float('inf')

print("Beginning Training and Evaluation Loop...")
for epoch in range(config.EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    total_train_samples = 0
    
    for inputs, labels in train_loader:
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE).unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)
        
    scheduler.step()
    epoch_train_loss = running_loss / total_train_samples
    
    # --- VALIDATION PHASE ---
    model.eval()
    validation_loss = 0.0
    total_val_samples = 0
    ground_truth_list = []
    model_scores_list = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE).unsqueeze(1)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            validation_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)
            
            # Map raw outputs to probability distributions
            probabilities = torch.sigmoid(outputs)
            
            ground_truth_list.extend(labels.cpu().numpy())
            model_scores_list.extend(probabilities.cpu().numpy())
            
    epoch_val_loss = validation_loss / total_val_samples
    
    # Calculate performance metrics
    epoch_eer, threshold_setting = compute_equal_error_rate(
        np.array(ground_truth_list), 
        np.array(model_scores_list)
    )
    
    print(f"Epoch [{epoch+1}/{config.EPOCHS}] -> "
          f"Train Loss: {epoch_train_loss:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} | "
          f"Val EER: {epoch_eer * 100:.2f}% | "
          f"Calibration Threshold: {threshold_setting:.4f}")
    
    # Metric Checkpoint Save
    if epoch_eer < best_eer:
        best_eer = epoch_eer
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_eer': best_eer,
            'forensic_threshold': threshold_setting,
            'config': {
                'sample_rate': config.SAMPLE_RATE,
                'n_mels': config.N_MELS,
                'target_samples': config.TARGET_SAMPLES
            }
        }, config.SAVE_PATH)
        print(f" -> Checkpoint Update: Saved model state with optimal EER: {best_eer * 100:.2f}%")

In [ ]:
def forensic_audio_analysis(file_path, model_checkpoint_path="forensic_audio_resnet.pt"):
    """
    Executes forensic evaluation on an isolated audio file path.
    Outputs classification metrics and system confidence rankings.
    """
    if not os.path.exists(model_checkpoint_path):
        raise FileNotFoundError(f"Selected model weights artifact path error: {model_checkpoint_path}")
        
    # Load serializations
    checkpoint = torch.load(model_checkpoint_path, map_location='cpu')
    model_meta = checkpoint['config']
    calibrated_threshold = checkpoint['forensic_threshold']
    
    # Reconstruct architectural pipeline
    eval_model = ForensicAudioResNet()
    eval_model.load_state_dict(checkpoint['model_state_dict'])
    eval_model.eval()
    
    # DSP Pipeline Transforms
    mel_transformer = T.MelSpectrogram(
        sample_rate=model_meta['sample_rate'],
        n_mels=model_meta['n_mels'],
        n_fft=1024,
        hop_length=256,
        f_min=20,
        f_max=8000
    )
    db_transformer = T.AmplitudeToDB()
    
    # Load raw target file
    waveform, sr = torchaudio.load(file_path)
    
    # Match dataset processing dimensions
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sr != model_meta['sample_rate']:
        waveform = T.Resample(sr, model_meta['sample_rate'])(waveform)
        
    # Padding / Truncation bounds execution
    target_len = model_meta['target_samples']
    if waveform.shape[1] > target_len:
        start = (waveform.shape[1] - target_len) // 2
        waveform = waveform[:, start:start + target_len]
    elif waveform.shape[1] < target_len:
        padding = target_len - waveform.shape[1]
        waveform = F.pad(waveform, (0, padding))
        
    # Transform feature extraction extraction execution
    log_mel = db_transformer(mel_transformer(waveform)).unsqueeze(0) # Add explicit batch dimension
    
    # Execute Model Prediction
    with torch.no_grad():
        logit = eval_model(log_mel)
        confidence_score = torch.sigmoid(logit).item()
        
    # Assessment execution against optimized EER boundary condition
    verdict = "DEEPFAKE / SPOOF" if confidence_score >= calibrated_threshold else "BONAFIDE / GENUINE"
    
    print("\n" + "="*50)
    print("           FORENSIC ANALYSIS REPORT           ")
    print("="*50)
    print(f"Target Source File:   {os.path.basename(file_path)}")
    print(f"Analysis Verdict:     {verdict}")
    print(f"Deepfake Probability: {confidence_score * 100:.2f}%")
    print(f"System Threshold:     {calibrated_threshold * 100:.2f}%")
    print("="*50 + "\n")
    
    return verdict, confidence_score

# Example invocation configuration (uncomment and supply file path to test execution)
# forensic_audio_analysis("/path/to/evidence/audio.flac")

FORENSIC-GRADE AUDIO DEEPFAKE & VOICE-CLONE DETECTION SYSTEM
Modality: Digital Signal Processing (DSP) & Deep Transfer Learning (ResNet-18)
Dataset Baseline: ASVspoof 2019 (Logical Access)
===================================================================================

1. EXECUTIVE SUMMARY & FORENSIC CONTEXT
In audio forensics, identifying synthesized text-to-speech (TTS) and voice conversion (VC) 
artifacts requires specialized metrics. Standard classification metrics like accuracy are 
highly deceptive due to severe class imbalance (benchmarks natively contain a vastly higher 
proportion of spoofed audio compared to bonafide records) and arbitrary thresholding (a 
fixed default threshold of 0.5 fails to adjust for fluctuating operational risks).

This system implements a professional biometric verification framework utilizing Log-Mel 
Spectrogram representations, an adapted ResNet-18 computer vision backbone, and Equal 
Error Rate ($EER$) optimization. It establishes an auditable, robust, and generalizable 
pipeline capable of identifying microscopic phase disruptions and spectral anomalies left 
behind by generative AI algorithms.

2. DATASET ARCHITECTURE: ASVspoof 2019 (LA)
The system is anchored on the ASVspoof 2019 Logical Access (LA) database, covering clean 
audio containing synthetic speech generated by 17 different speech synthesis and voice 
conversion systems (designated as attacks A01 through A19).
Protocol Structure: [Speaker_ID] [Audio_File_Name] [-] [Attack_System_ID] [Target_Label]
- Bonafide (Genuine): Labeled as 'bonafide', mapped to class 0.
- Spoof (Deepfake): Labeled as 'spoof', mapped to class 1.

3. DIGITAL SIGNAL PROCESSING (DSP) PIPELINE
Raw acoustic pressure waves are transformed into time-frequency log-spectral representations:
Raw Audio (.flac) ──► Mono Downmix ──► Resample (16kHz) ──► Windowing/Slicing (4s) 
                          │
                          ▼
            Discrete Fourier Transform (STFT)
                          │
                          ▼
             Mel-Frequency Scale Mapping
                          │
                          ▼
             Log-Scale Decibel Transformation (dB)
                          │
                          ▼
         SpecAugment (Frequency & Time Masking)

Mathematical Foundations:
- The Mel Scale Transformation: Maps physical frequency f (in Hz) to perceptual Mel scale m:
  $$m = 2595 \cdot \log_{10}\left(1 + \frac{f}{700}\right)$$
- Windowing & Fixed Length Constraint: Audio is resampled uniformly to 16 kHz and forced to 
  a static length of 4.0 seconds (64,000 samples).
- Log-Mel Conversion: Amplitudes are compressed into decibel units ($dB$) via:
  $$S_{dB} = 10 \cdot \log_{10}\left(\frac{P}{P_{ref}}\right)$$
- SpecAugment Regularization: Frequency and time blocks are randomly masked during training 
  to force the model to learn decentralized diagnostic indicators instead of memorizing 
  localized acoustic environments.

4. DEEP LEARNING MODEL ARCHITECTURE
Utilizes an adapted ResNet-18 computer vision backbone:
- Input Layer (conv1): Modified from 3-channel (RGB) to 1-channel (Grayscale). Weights are 
  initialized by cloning and scaling the first pre-trained channel layer.
- Feature Extraction: 4 Residual Blocks with skip connections to mitigate vanishing gradients.
- Global Pool: AdaptiveAvgPool2d(1, 1) condenses variable temporal maps into a single vector.
- Forensic Head: Integrated Dropout(p=0.5) layer projecting to a single raw logit output.

5. OPTIMIZATION & STATISTICAL CALIBRATIONS
- Position-Weighted Loss Engine: Employs BCEWithLogitsLoss with a positive weighting index (w = 9.0) 
  to manage structural skew:
  $$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^N \left[ w \cdot y_i \cdot \log \sigma(x_i) + (1 - y_i) \cdot \log(1 - \sigma(x_i)) \right]$$
- Optimizer & Scheduler: AdamW (weight decay = $10^{-4}$) combined with CosineAnnealingLR.

6. THE FORENSIC METRIC: EQUAL ERROR RATE (EER)
Forensic acoustic identification systems rely on the Equal Error Rate ($EER$), tracking the 
precise operational threshold where False... Acceptance Rate ($FAR$) equals False Rejection Rate ($FRR$):
  $$FAR(\theta) = FRR(\theta)$$
The script saves the exact model state along with this Calibrated Calibration Threshold ($\theta$) 
to ensure reliable real-world forensic execution boundaries.
"""